# IndicF5 voice-cloning proof of concept

This notebook runs the local CLI in Google Colab. Use a GPU runtime when available.

Only use a reference recording with the speaker's informed consent. Do not commit or share the recording, exact transcript, Hugging Face token, or generated audio. The transcript cell below stays private to the current Colab runtime.

## 1. Clone the implementation

If the repository is private, enter a GitHub fine-grained token with read-only access to this repository when prompted. The token is passed to Git only through a temporary `GIT_ASKPASS` environment and is not written to the notebook, clone URL, or repository. Leave the prompt blank only for a public repository.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/iammoin/kpv.git"
REPO_REF = "main"
REPO_DIR = Path("/content/kpv")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

github_token = getpass(
    "GitHub fine-grained token (leave blank only if the repository is public): "
)
clone_env = os.environ.copy()
if github_token:
    askpass = Path("/tmp/kpv_git_askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) printf '%s' 'x-access-token' ;;\n"
        "  *) printf '%s' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n",
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    clone_env.update(
        {
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "GITHUB_TOKEN": github_token,
        }
    )

try:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env=clone_env,
    )
finally:
    github_token = None
    clone_env.pop("GITHUB_TOKEN", None)

%cd /content/kpv/voice-clone-poc
Path("samples").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

## 2. Install Colab dependencies

The repository requirements file is pinned for the Windows CPU workstation. Colab uses its Linux runtime and preinstalled PyTorch, so this cell installs the official IndicF5 package plus Colab-compatible runtime dependencies instead.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
%pip install -q "numpy<=1.26.4" "transformers<4.50" soundfile torchcodec "git+https://github.com/AI4Bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7"

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Using CPU; generation may be slow.')

## 3. Authenticate and upload the reference

First accept access to [ai4bharat/IndicF5](https://huggingface.co/ai4bharat/IndicF5). The login prompt below is interactive and does not save a token in the notebook.

Upload a clean 20–60 second WAV containing one consenting speaker.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
wav_files = [name for name in uploaded if name.lower().endswith('.wav')]
if len(wav_files) != 1:
    raise ValueError('Upload exactly one WAV reference file.')
shutil.copyfile(wav_files[0], 'samples/spouse_hinglish.wav')
print('Reference copied to samples/spouse_hinglish.wav')

In [ ]:
import json

REFERENCE_TEXT = """PASTE THE EXACT WORDS SPOKEN IN THE UPLOADED WAV HERE"""
if REFERENCE_TEXT.startswith('PASTE THE EXACT'):
    raise ValueError('Replace REFERENCE_TEXT with the exact transcript before generating.')

Path('reference_texts.local.json').write_text(
    json.dumps({'hinglish': REFERENCE_TEXT}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Private transcript written only to the current Colab runtime.')

## 4. Generate one test phrase

The first valid run downloads the model and may take several minutes on CPU.

In [ ]:
!python app.py --text "Aap currently koi medicines le rahe hain?" --reference hinglish --output outputs/test_01.wav

In [ ]:
import soundfile as sf
info = sf.info('outputs/test_01.wav')
print(info)

## 5. Generate the five evaluation phrases

Run this only after listening to `test_01.wav` and confirming that the output is a real waveform rather than an artifact.

In [ ]:
!python app.py --generate-tests --reference hinglish

In [ ]:
from google.colab import files
files.download('outputs/test_01.wav')

## 6. Manual evaluation

Listen to all five files and enter 1–5 ratings in `evaluation.md` locally. Do not claim voice similarity or pronunciation quality from file creation alone.